# Experiment 2 — HR Policy Bot | LoRA Fine-Tuning


###What I'm experimenting with

In this project, I'm applying LoRA fine-tuning to TinyLlama again, this time using examples based on HR policy questions.

The goal is to see how the model's responses change when the training data is focused on a specific domain, such as:

leave policies
reimbursements
onboarding
employee benefits
workplace conduct
What is different from Project 1?

This project builds on the previous experiment, but uses a different dataset and training configuration.

The training dataset contains 20 examples instead of 10.
The training settings are slightly different, so I can compare the training behaviour with the previous experiment.
The main focus is domain adaptation — observing whether training on HR-related examples makes the model's responses more aligned with that domain.

I'll use the same general LoRA fine-tuning workflow from the first project and compare the results.

## Cell 1 — Install libraries

In [ ]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.7 MB/s eta 0:00:00


## Cell 2 — Imports + GPU check

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import torch

print("GPU available:", "YES —", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO — go to Runtime > Change Runtime Type > T4 GPU")

GPU available: YES — Tesla T4


## Cell 3 — Training data

The training dataset contains **20 HR policy instruction-response pairs**.

The examples follow a consistent structure and are written in a formal, policy-oriented style. Keeping the examples consistent should help the fine-tuning process adapt the model's responses toward the intended domain and format.

For this experiment, I'm keeping the dataset relatively small so I can observe how the change in domain and dataset size affects the model's behaviour.


In [ ]:
training_examples = [
    {
        "instruction": "How many casual leaves am I entitled to per year?",
        "response": "As per company policy, all full-time employees are entitled to 12 casual leaves per calendar year. These cannot be carried forward to the next year and are credited at the start of each January. Probationary employees receive casual leaves on a pro-rata basis from their date of joining."
    },
    {
        "instruction": "What is the process to apply for maternity leave?",
        "response": "Maternity leave must be applied at least 30 days before the expected date of delivery. You are entitled to 26 weeks of paid maternity leave as per the Maternity Benefit Act. Please submit Form ML-01 to your HR Business Partner along with the doctor's certificate confirming the expected due date."
    },
    {
        "instruction": "Can I work from home permanently?",
        "response": "Permanent work from home arrangements require approval from your Department Head and HR. The decision is based on the nature of your role, team requirements, and performance history. You may submit a formal request through the HR portal under the 'Flexible Work Arrangement' section. All approvals are reviewed quarterly."
    },
    {
        "instruction": "How do I claim travel reimbursement for a client visit?",
        "response": "To claim travel reimbursement, submit your expenses within 7 working days of the trip via the expense management portal. Attach original receipts for all claims above Rs. 500. Approved modes of travel for client visits include flights (economy class), trains, and cabs booked through the company travel desk. Claims submitted after 7 days may not be processed."
    },
    {
        "instruction": "What happens to my earned leaves if I don't use them?",
        "response": "Earned leaves can be accumulated up to a maximum of 45 days. Leaves beyond this cap will lapse at the end of the financial year. Employees may also opt for leave encashment for up to 15 earned leaves per year, subject to management approval. This can be requested during the annual leave review cycle in March."
    },
    {
        "instruction": "What is the notice period for resignation?",
        "response": "The standard notice period is 60 days for all confirmed employees. During your probation period, the notice period is 30 days. Notice period buyout is available subject to business requirements and manager approval. Your relieving letter and full and final settlement will be processed within 30 days of your last working day."
    },
    {
        "instruction": "How do I update my bank account details for salary?",
        "response": "Bank account updates must be submitted at least 10 working days before the salary processing date to ensure timely credit. Log in to the HR self-service portal, navigate to 'My Profile', and select 'Bank Details'. Attach a cancelled cheque or bank statement as proof. Changes are verified by payroll before activation."
    },
    {
        "instruction": "Am I eligible for a performance bonus this year?",
        "response": "Performance bonuses are awarded to employees who have completed at least 6 months of service in the appraisal cycle and have received a performance rating of 'Meets Expectations' or above. Bonus amounts are determined by your band, rating, and the company's overall financial performance. Bonus letters are shared by HR in April each year."
    },
    {
        "instruction": "What is the policy on internet usage at work?",
        "response": "Company internet resources are to be used for business purposes only. Accessing non-work related streaming platforms, gaming sites, or social media for personal use during work hours is not permitted on company networks. All internet activity on company devices is monitored. Violation of this policy may result in disciplinary action as per the Code of Conduct."
    },
    {
        "instruction": "How do I report a workplace harassment complaint?",
        "response": "All harassment complaints are handled with strict confidentiality under the Prevention of Sexual Harassment (POSH) policy. You may report an incident by emailing the Internal Complaints Committee at posh@company.com or by speaking directly to your HR Business Partner. A formal inquiry will be initiated within 7 working days of the complaint being registered. Retaliation against complainants is strictly prohibited."
    },
    {
        "instruction": "What benefits are included in my health insurance?",
        "response": "The company's group health insurance covers you, your spouse, and up to two dependent children. Coverage includes hospitalisation, daycare procedures, pre and post-hospitalisation expenses up to 30 and 60 days respectively, and maternity benefits. The annual sum insured is Rs. 5 lakhs per family. You may opt for a top-up plan at a subsidised premium during the annual enrolment window."
    },
    {
        "instruction": "Can I take a sabbatical leave?",
        "response": "Sabbatical leave is available to employees with more than 5 years of continuous service. You may apply for an unpaid sabbatical of up to 6 months for purposes such as higher education, personal development, or caregiving. Approval is subject to business continuity requirements and must be applied for at least 60 days in advance through the HR portal."
    },
    {
        "instruction": "What is the process for internal job transfers?",
        "response": "Internal transfers are encouraged and can be initiated after completing 18 months in your current role. Browse open positions on the internal careers portal and apply directly. Inform your current manager before applying as a professional courtesy, though their approval is not mandatory. HR will coordinate between both teams to manage the transition timeline, typically 30 to 45 days."
    },
    {
        "instruction": "How is my provident fund contribution calculated?",
        "response": "Your Provident Fund contribution is 12% of your basic salary, matched equally by the employer. Both contributions are deposited monthly with the Employee Provident Fund Organisation (EPFO). You can view your PF balance and download your passbook by logging into the EPFO member portal using your UAN number. Your UAN is available in your salary slip and the HR portal."
    },
    {
        "instruction": "What is the dress code policy?",
        "response": "The company follows a business casual dress code from Monday to Thursday. Fridays are designated casual days where smart casuals are acceptable. When attending client meetings or office events, formal business attire is expected regardless of the day. Clothing with offensive graphics, slogans, or torn clothing is not permitted at any time on company premises."
    },
    {
        "instruction": "How do I apply for a study leave?",
        "response": "Study leave may be granted for courses or examinations directly relevant to your current role or career development at the company. You are eligible for up to 5 paid study leave days per year. Submit your application through the HR portal at least 14 days before the required date, attaching proof of enrollment or examination schedule. Approval rests with your line manager and HR."
    },
    {
        "instruction": "What happens if I am absent without approval?",
        "response": "Unapproved absences are marked as Loss of Pay (LOP) for the days concerned. If an employee is absent without communication for more than 5 consecutive working days, it may be treated as voluntary abandonment of employment, which can lead to termination. It is important to always inform your manager and apply for leave in advance or on the same day in case of emergencies."
    },
    {
        "instruction": "Is there a referral bonus for recommending a candidate?",
        "response": "Yes, the employee referral program offers a bonus of Rs. 25,000 for successful referrals at junior and mid levels, and Rs. 50,000 for senior level positions. The referred candidate must complete 90 days of employment before the bonus is paid out. Submit referrals through the HR portal under 'Refer a Friend'. Self-referrals and referrals of immediate family members are not eligible."
    },
    {
        "instruction": "Can I opt out of the National Pension Scheme?",
        "response": "Enrollment in the National Pension Scheme (NPS) is optional for employees below the age of 60. If you wish to opt in or out, you may do so during the annual benefits window in April. Changes made outside this window are not processed until the next cycle. Contact your HR Business Partner or the payroll team for guidance on NPS contribution rates and tax benefits."
    },
    {
        "instruction": "What is the policy for using company laptops for personal use?",
        "response": "Company-issued laptops are provided for business use only. Limited personal use is tolerated provided it does not interfere with work, consume excessive bandwidth, or involve downloading unauthorised software. Installation of any software not approved by the IT department is strictly prohibited. Company laptops must not be used for illegal activities. All data on company devices is subject to monitoring and audit."
    },
]

def format_example(ex):
    return {"text": f"""<|system|>
You are an HR policy assistant for a large enterprise. Answer employee questions accurately, professionally, and based strictly on company policy.</s>
<|user|>
{ex['instruction']}</s>
<|assistant|>
{ex['response']}</s>"""}

dataset = Dataset.from_list([format_example(ex) for ex in training_examples])

print(f"Training examples loaded: {len(dataset)}")
print("-" * 50)
print(dataset[0]["text"])

Training examples loaded: 20
--------------------------------------------------
<|system|>
You are an HR policy assistant for a large enterprise. Answer employee questions accurately, professionally, and based strictly on company policy.</s>
<|user|>
How many casual leaves am I entitled to per year?</s>
<|assistant|>
As per company policy, all full-time employees are entitled to 12 casual leaves per calendar year. These cannot be carried forward to the next year and are credited at the start of each January. Probationary employees receive casual leaves on a pro-rata basis from their date of joining.</s>


## Cell 4 — Loading model with 4-bit quantization

In [ ]:
from peft import PeftModel

# Loading the base model fresh
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load trained LoRA adapter on top of it
print("Loading your trained LoRA adapter...")
model = PeftModel.from_pretrained(base_model, "./tinyllama-hr-bot-final")
model.eval()

print("Done! Your trained model is ready.")

Loading base model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading your trained LoRA adapter...
Done! Your trained model is ready.


## Cell 5 — ingh LoRA

This follows the same LoRA approach as Project 1, but I'm using **`r=16`** instead of `r=8`.

A higher rank increases the capacity of the LoRA adapters by adding more trainable parameters. In this experiment, I'm using the higher rank to see whether it affects the model's adaptation to the more structured HR policy dataset.

I'll compare the training behaviour and model responses with the results from Project 1.


In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


## Cell 6 — Training

During training, I'll monitor the loss and how it changes across the training steps.

A decreasing loss generally indicates that the model is fitting the training examples, but the loss value needs to be considered together with the model's actual responses.

Since this experiment uses a different dataset and LoRA rank from Project 1, I'll compare the training behaviour and final responses rather than relying on a fixed loss threshold.

In [ ]:
training_args = SFTConfig(
    output_dir="./tinyllama-hr-bot",
    num_train_epochs=30,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    bf16=True,
    logging_steps=5,
    save_steps=100,
    max_length=512,
    dataset_text_field="text",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer
)

print("Training started — watch the loss go down...")
print("-" * 50)
trainer.train()
print("-" * 50)
print("Training complete!")

Adding EOS to train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Training started — watch the loss go down...
--------------------------------------------------


Step,Training Loss
5,1.981778
10,1.611557
15,1.276468
20,1.039346
25,0.944826
30,0.848811
35,0.748646
40,0.659695
45,0.576919
50,0.500336


--------------------------------------------------
Training complete!


## Cell 7 — Saving the LoRA adapter

In [ ]:
trainer.model.save_pretrained("./tinyllama-hr-bot-final")
tokenizer.save_pretrained("./tinyllama-hr-bot-final")
print("Adapter saved to: ./tinyllama-hr-bot-final")

Adapter saved to: ./tinyllama-hr-bot-final


## Cell 8 — Test the HR Bot

I'm testing the fine-tuned model with HR-related questions to see how well it responds after training.
The goal is to observe how the model behaves after domain-specific fine-tuning and identify where the approach works well or where it still has limitations.

In [ ]:
def get_hr_response(question, max_new_tokens=200):
    prompt = f"""<|system|>
You are an HR policy assistant for a large enterprise. Answer employee questions accurately, professionally, and based strictly on company policy.</s>
<|user|>
{question}</s>
<|assistant|>
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.6,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Only decode the NEW tokens — not the prompt we already sent
    new_tokens = outputs[0][input_length:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()


test_questions = [
    "How many sick leaves do I get in a year?",
    "What is the process to apply for paternity leave?",
    "Can I use my earned leaves during notice period?",
]

print("HR Bot Responses on Unseen Questions")
print("=" * 60)
for q in test_questions:
    print(f"\nEmployee: {q}")
    print(f"HR Bot: {get_hr_response(q)}")
    print("-" * 60)

HR Bot Responses on Unseen Questions

Employee: How many sick leaves do I get in a year?
HR Bot: Federal law requires up to two consecutive years of work experience before being eligible for paid sick days. You may ask for up to four weeks of paid sick leave per annually specified by the manager. However, emergency medical procedures demanded outside normal working hours or at midnight or beyond weekends carries a chargeable fee. Contact your HR business partner for guidance on other scenarios.
------------------------------------------------------------

Employee: What is the process to apply for paternity leave?
HR Bot: Pronunced father or mother must submit Form F1 (https://forms.office.com/r/?query=FormF1PaternityLeave) at least 24 hours before the expected date of delivery to claim up to 5 full working days paid maternal leave per birth. All leaves must be restored prior to the birth of the child. Probationary employees may also opt for flexible work arrangement up until pregnancy

##Observation:

### Observation

The fine-tuned model is producing responses that follow an HR-support style, but the responses are not consistently reliable.

Some answers contain specific policy details that were not necessarily supported by the training examples, including invented procedures, eligibility conditions, and form information. This suggests that the fine-tuning helped shape the **style and domain**, but did not make the model a reliable source of HR policy information.

This is an important limitation of the experiment: fine-tuning alone does not guarantee factual accuracy or prevent hallucinations. In a real HR assistant, the model would need access to an authoritative policy source, such as a knowledge base or RAG system, to ground its answers in current company policies.


## Cell 9 — Random question

In [ ]:
your_question = "How do I apply for a study leave??"

print(f"Employee: {your_question}")
print(f"\nHR Bot: {get_hr_response(your_question)}")

Employee: How do I apply for a study leave??

HR Bot: Study leaves can be applied for during the annual calendar year as well as summer semester breaks. You may request up to three study leaves per academic year under special circumstances or courses directly related to your professional expertise. Submission of approved studies allowance forms is submitted by the Financial Assistance Unit. Applications must be submitted at least 14 business days before the desired start date.


##Observation(random question)

The model continues to produce structured, formal HR-style responses when given a new question.

However, the answer also contains very specific details such as three study leaves per academic year, a Financial Assistance Unit, and a 14-business-day requirement. These details should not automatically be treated as factual simply because the response is confident and well-structured.

This reinforces the limitation seen in the previous test: the fine-tuned model has adapted to the style and domain, but fine-tuning on a small dataset is not enough to ensure factual accuracy.

---
#Evaluation observation

The fine-tuned model was able to produce responses with a consistent HR-support style, but some responses contained specific policy details that were not reliably grounded in the provided examples.

This was the main takeaway from the experiment: LoRA fine-tuning can adapt a model's behaviour and style to a domain, but fine-tuning alone is not a substitute for grounding the model in an authoritative source.

A real HR policy assistant would benefit from a RAG or other retrieval-based approach so that responses can be based on the actual policy documents rather than relying only on what the model learned during fine-tuning.

##Project conclusion

This experiment helped me compare two small-scale LoRA fine-tuning setups and understand the difference between learning a domain-specific response style and providing reliable domain-specific information.